In [1]:
#from Run_Model import *


In [2]:
# -------- import & helpers ------------------------------------
from pathlib import Path
import json, os
from dotenv import load_dotenv
from langchain.schema import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_text_splitters import RecursiveCharacterTextSplitter, RecursiveJsonSplitter
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from konlpy.tag import Okt


def load_env_variables_for_Local():
    """
    .env 파일에서 환경 변수를 로드합니다.
    """
    # .env 파일의 경로를 지정합니다.
    # 현재 스크립트의 상대 경로로 설정합니다.
    # load_dotenv(dotenv_path='../../.env')
    # 절대 경로로 설정합니다.
    load_dotenv(dotenv_path='../../.env')

    os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
    os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT')
    os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT')
    os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING')
    os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')


def load_env_variables_for_Colab():
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT2')
    os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT2')
    os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
    os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY2')


# ---------- tokenizers ----------------------------------------
okt = Okt()


def len_okt(t):  return len(okt.morphs(t))


def okt_tokenize(t): return okt.morphs(t)


# ---------- 1. 로더 -------------------------------------------
def load_files(folder: str, kind: str) -> list[Document]:
    docs: list[Document] = []

    if kind in ("json", "all"):
        for p in Path(folder).rglob("*.json"):
            with p.open(encoding="utf-8") as f:
                txt = json.dumps(json.load(f), ensure_ascii=False, indent=2)
            docs.append(Document(page_content=txt, metadata={"source": p.as_posix()}))

    if kind in ("txt", "all"):
        docs.extend(DirectoryLoader(folder, glob="**/*.txt").load())

    if not docs:
        raise ValueError("파일을 찾지 못했습니다.")

    print(f"파일 {len(docs)}개 로드 완료")
    print(f"파일 종류: {kind}")
    return docs


# ---------- 2. 스플리터 ---------------------------------------
def split_docs(docs: list[Document], kind: str,
               chunk: int = 1000, overlap: int = 100):
    if kind == "json":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk, chunk_overlap=overlap, length_function=len_okt
        )
        flat = [d.page_content for d in docs]
        return splitter.create_documents(flat)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk, chunk_overlap=overlap, length_function=len_okt
    )
    return splitter.split_documents(docs)


# ---------- 3. 임베딩 & DB ------------------------------------
# 1) 임베딩 ─────────────────────────────────────────────
# pip install -U langchain-huggingface
from langchain_huggingface import HuggingFaceEmbeddings


def load_embed(device: str, model_name: str):
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},
    )


# 2) Chroma DB ─────────────────────────────────────────
from langchain_chroma import Chroma
from pathlib import Path

from pathlib import Path
from langchain_chroma import Chroma


def get_db(texts, embed, persist_dir: str):
    persist_path = Path(persist_dir)
    persist_path.mkdir(parents=True, exist_ok=True)

    if any(persist_path.iterdir()):
        print("기존 Chroma DB 로드")
        db = Chroma(persist_directory=str(persist_path), embedding_function=embed)
    else:
        print("새 Chroma DB 생성")
        db = Chroma.from_documents(
            texts,
            embed,
            persist_directory=str(persist_path)
        )

    return db


# 사용 예
# db = get_db(documents, embedding_fn, "my_chroma_db")
# 이후 추가 삽입 등 변경이 있으면:
# db.add_documents(new_docs)
# db.persist()


# ---------- 4. Retriever --------------------------------------
def build_retriever(mode: int, k: int, db, docs):
    vec = db.as_retriever(search_kwargs={"k": k})
    bm = BM25Retriever.from_documents(docs, preprocess_func=okt_tokenize);
    bm.k = k
    if mode == 1: return vec
    if mode == 2: return bm
    if mode == 3: return EnsembleRetriever(retrievers=[vec, bm], weights=[0.5, 0.5])
    raise ValueError("retriever_num 은 1~3")


# ---------- 5. LLM --------------------------------------------
def load_llm(engine: int, backend: int):
    if engine == 1:
        name = "gpt-4o-mini"
    elif engine == 2:
        name = "gemma3:4b"
    elif engine == 3:
        name = "quen3:4b"
    else:
        raise ValueError("engine_num 은 1~3")

    if backend == 1:
        return ChatOpenAI(model=name, temperature=0,
                          streaming=True,
                          callbacks=[StreamingStdOutCallbackHandler()],
                          )
    elif backend == 2:
        return ChatOllama(model=name, temperature=0,
                          streaming=True,
                          callbacks=[StreamingStdOutCallbackHandler()],
                          )
    else:
        raise ValueError("1,2번 중 선택해 주세요.")


# ---------- 6. Chain ------------------------------------------
prompt_content = """
            You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.

            Answer in Korean.

            #Context:
            {context}
            """

PROMPT = ChatPromptTemplate.from_messages([

    (
        "system",
        prompt_content,
    ), ("human", "{question}"),

])


def build_chain(retriever, llm):
    format_docs = lambda ds: "\n\n".join(d.page_content for d in ds)
    chain = {
                "context": retriever | RunnableLambda(format_docs),
                "question": RunnablePassthrough(),
            } | PROMPT | llm | StrOutputParser()
    return chain

from langchain.chains import RetrievalQA

def build_qa_chain(retriever, llm):
    prompt = ChatPromptTemplate.from_messages([
        ("system", prompt_content),
        ("human", "{query}")
    ])

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",                 # 또는 "map_reduce", "refine" 등
        retriever=retriever,
        return_source_documents=False,
        chain_type_kwargs={"prompt": prompt}
    )
    return qa_chain



In [3]:
# ----- ① 환경 변수 로드 ---------------------------------
if os.path.exists("../../.env"):
    load_env_variables_for_Local()  # 로컬 실행
else:
    load_env_variables_for_Colab()  # Colab 실행

In [4]:
kind = input("파일 형식(json/txt): ").strip()
folder = "../../Data_Files"
docs = load_files(folder, kind)

파일 1개 로드 완료
파일 종류: json


In [5]:
chunk = int(input("chunk: "))
texts = split_docs(docs, kind, chunk)

In [6]:
device = {1: "mps", 2: "cuda", 3: "cpu"}[int(input("디바이스(1:mps/2:cuda/3:cpu): "))]
embed = load_embed(device, "nlpai-lab/KURE-v1")
db = get_db(texts, embed, f"./{kind}_{chunk}")

기존 Chroma DB 로드


In [7]:
mode = int(input("retriever (1 vec / 2 bm25 /3 ensemble): "))
k = int(input("k 개수를 입력해 주세요: "))
retr = build_retriever(mode, k=k, db=db, docs=texts)

In [12]:
llm_model = int(input("LLM 모델 번호(1: gpt-4o-mini / 2: gemma3:4b / 3: quen3:4b): "))
llm = load_llm(llm_model_num := llm_model, backend=int(input("LLM (1 openai / 2 ollama): ")))
chain = build_chain(retr, llm)

In [13]:
while True:
    q = input("\n질문(종료 exit): ")
    if q.lower() == "exit":
        break
    print(chain.invoke(q))

최근 개발자 채용 공고 개수는 다음과 같이 변화하고 있습니다.

**최신 데이터 (2024년 5월 24일 기준):**

*   **잡코리아:** 10,387개
*   **사람인:** 8,783개
*   **Indeed:** 11,373개
*   **링크드인:** 10,000개 이상 (실시간 검색 필요)

**전반적인 추세:**

*   **급증하는 채용 공고:** 최근 몇 달 동안 개발자 채용 공고는 꾸준히 증가하는 추세입니다. 이는 IT 산업 전반의 성장과 함께 개발 인력에 대한 수요가 증가하고 있기 때문입니다.
*   **주요 채용 분야:**
    *   **프론트엔드 개발:** 3,000개 이상
    *   **백엔드 개발:** 2,000개 이상
    *   **풀스택 개발:** 3,000개 이상
    *   **모바일 앱 개발:** 2,000개 이상
    *   **데이터 엔지니어:** 1,000개 이상
    *   **AI/머신러닝 개발:** 1,000개 이상

**참고:**

*   위 수치는 실시간으로 변동될 수 있습니다.
*   채용 공고 플랫폼별로 수치가 다를 수 있습니다.
*   채용 공고의 종류 (정규직, 계약직, 인턴 등)에 따라도 수치가 달라질 수 있습니다.

**채용 공고 검색 사이트:**

*   [잡코리아](https://www.jobkorea.co.kr/)
*   [사람인](https://www.saramin.co.kr/)
*   [Indeed](https://kr.indeed.com/)
*   [링크드인](https://www.linkedin.com/)

**팁:**

*   여러 채용 공고 플랫폼을 동시에 검색하여 다양한 정보를 얻으세요.
*   자신의 기술 스택과 경력에 맞는 공고를 선택하세요.
*   채용 공고의 상세 내용을 꼼꼼히 확인하고, 지원 자격 및 우대 조건 등을 확인하세요.

더 궁금한 점이 있으시면 언제든지 질문해주세요.최근 개발자 채용 공고 개수는 다음과 같이 변화하고 있습니다.

**최신 데이터 

KeyboardInterrupt: Interrupted by user